# Cross-Lingual Knowledge Distillation for Low-Resource Filipino Sentiment Analysis

Group 1

Purpose: Install required libraries for the entire project.

Why: Keras-hub provides XLM-RoBERTa models, kagglehub enables data loading from Kaggle datasets, and these packages are not pre-installed in the environment.



## Project Goal

Develop an efficient Filipino sentiment analysis model through knowledge distillation that achieves high accuracy, fast inference, and small model size. The goal is to leverage a large multilingual teacher model to train a smaller, efficient student model suitable for resource-constrained devices while maintaining acceptable performance on low-resource Filipino language data.

## What We've Done So Far (Summary)

### Phase 1: Data Preparation
Collected 1,000 Lazada product reviews and created binary sentiment labels based on ratings (≤3 = negative, >3 = positive). After removing 211 empty reviews, 789 reviews remained. Data was split into training (60%, 593 reviews), validation (20%, 198 reviews), and test (20%, 198 reviews) sets. All reviews were tokenized to 128-token sequences using the XLM-RoBERTa tokenizer.

### Phase 2: Teacher Model Development
Selected XLM-RoBERTa-Base (559M parameters, 12 transformer layers) as the teacher model. Trained on hard labels for 10 epochs using cross-entropy loss. Achieved 84.85% test accuracy with precision of 84.62%, recall of 86.27%, and F1-score of 85.44%. Inference time was 0.764 seconds for 198 samples.

### Phase 3: Soft Target Generation
Extracted probability distributions from the trained teacher model on all data splits. These soft targets preserve the teacher's decision confidence and learned boundaries. Saved soft targets as NumPy arrays for student training.

### Phase 4: Student Model Development
Selected DistilBERT-Base-Multilingual (335M parameters, 6 transformer layers) as the student model. Resized vocabulary from 119K to 250K tokens to match the teacher. Implemented dual-loss distillation with 60% KL divergence loss and 40% cross-entropy loss. Used temperature of 2.0 for soft target softening. Trained for 15 epochs with AdamW optimizer and learning rate scheduling.

### Phase 5: Model Evaluation and Comparison
Tested both models on the held-out test set. Student achieved 77.78% accuracy (91.6% of teacher's accuracy), precision of 82.95%, recall of 71.57%, and F1-score of 76.84%. Student inference time was 0.406 seconds (1.88x faster than teacher). Model size was reduced by 40% (335M vs 559M parameters).

### Phase 6: Model Persistence
Exported both teacher and student models with tokenizers in standard HuggingFace format. Created reusable inference functions for sentiment prediction on new samples.


## Key Results

| Metric | Teacher | Student | Change |
|--------|---------|---------|--------|
| Accuracy | 84.85% | 77.78% | -7.07% |
| Precision | 84.62% | 82.95% | -1.67% |
| Recall | 86.27% | 71.57% | -14.7% |
| F1-Score | 85.44% | 76.84% | -8.6% |
| Inference Time | 0.764s | 0.406s | 1.88x faster |
| Parameters | 559M | 335M | 40% smaller |
| Knowledge Retention | 100% | 91.6% | Excellent |


## What We Learned

Dual-loss distillation approach (KL divergence plus cross-entropy) is effective for low-resource languages. Temperature scaling at 2.0 enables smooth learning from soft targets. The student retained 91.6% of teacher accuracy while achieving 40% model compression and 1.88x speedup. However, the student struggles with complex linguistic phenomena like negation and sarcasm due to reduced model capacity. The trade-off between speed/size and accuracy is acceptable for deployment-focused applications.


## Current State

The implementation is complete with a functional knowledge distillation pipeline from data preparation through evaluation. Both teacher and student models are trained and exported. Code is annotated with markdown descriptions for each step. Next step is UI development

In [1]:
# Install environment dependencies
!pip install keras-hub keras kagglehub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 319.9/319.9 kB 6.8 MB/s eta 0:00:0000:01
  Attempting uninstall: protobuf
    Found existing installation: protobuf 6.33.0
    Uninstalling protobuf-6.33.0:
      Successfully uninstalled protobuf-6.33.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.12.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-cloud-translate 3.12.1 requires protobuf!=3.20.0,!=3.20.1,!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<5.0.0dev,>=3.19.5, but you have protobuf 5.29.5 which is incompatible.
ray 2.51.1 requires click!=8.3.0,>=7.0, but you have click 8.3.0 which is incompatible.
bigframes 2.12.0 requires rich<14,>=12.4.4, but you have rich 14.2.0 which is incompatible.
pydrive2 1.21.3 requires cryptography<44, but you have cryptography 46.0.3 which is incompatible.
pydrive2 1.2

In [2]:
"""Import libraries"""

# Data Manipulation and Utilities
import pandas as pd
import torch
import numpy as np  # Often implicitly needed with PyTorch/TensorFlow, good to include if used
import json

# Machine Learning / Deep Learning Frameworks
import keras
import keras_hub
from tensorflow.keras.utils import to_categorical
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from torch.utils.data import DataLoader, TensorDataset
from torch.optim import AdamW

# Model Selection / Sklearn Utilities
from sklearn.model_selection import train_test_split

# External Model Hubs
import kagglehub

# Visualization
import plotly.graph_objects as go
from plotly.subplots import make_subplots

## II. About the Dataset

1k product reviews scraped from Lazada.

## Load and Label Data

Load 1,000 Lazada reviews from JSON file, create binary sentiment labels based on ratings, and clean empty reviews. We need raw data prepared with sentiment labels to train the teacher model. Binary labeling (rating > 3 = positive) converts numerical ratings to sentiment classes. Cleaning removes invalid data that would cause errors during training.

In [4]:
# Load the Data
file_name = '/kaggle/input/lazada-reviews/reviews.json'
reviews = []
with open(file_name, errors='ignore') as f:
    reviews = json.load(f)

df = pd.DataFrame(reviews)

# Remove rows with empty string
initial_size = len(df)
df = df[df['review'] != ''].copy()
rows_removed = initial_size - len(df)
print(f"Removed {rows_removed} rows where the 'review' field was an empty string.")
# ----------------------------------------------------

df["sentiment"] = "negative"
# Assign 'positive' where the rating is greater than 3
df.loc[df["rating"] > 3, "sentiment"] = "positive"

print('Final sample size:', len(df))



Removed 12 rows where the 'review' field was an empty string.
Final sample size: 989


### Convert to Lists

Convert DataFrame columns to Python lists for easier manipulation and compatibility with PyTorch DataLoaders. PyTorch models expect data as lists or tensors, not pandas Series. Converting to lists also removes DataFrame indexing overhead.

In [5]:
# Convert to list
x = df['review'].tolist()
y = (df['sentiment']=='positive').astype(int).tolist()

### Data Splitting

Split data into training (60%), validation (20%), and test (20%) sets using stratified sampling. Stratified splitting maintains class balance in each subset, preventing class imbalance bias. Training set trains the model, validation set tunes hyperparameters, and test set provides unbiased final evaluation.

In [6]:
"""Data Splitting"""
# First split: 80% train+val, 20% test
x_temp, x_test, y_temp, y_test = train_test_split(x, y, test_size=0.2, random_state=42)

# Second split: 75% train, 25% val (of the temp set)
x_train, x_val, y_train, y_val = train_test_split(x_temp, y_temp, test_size=0.25, random_state=42)

## Teacher Model Setup and Training

Load XLM-RoBERTa-Base, tokenize data, create DataLoaders, and train the teacher model for 10 epochs using supervised learning. The teacher model learns sentiment patterns from hard labels (ground truth). A strong teacher (84%+ accuracy) is essential for effective knowledge distillation. This model will later generate soft targets to guide student training.

In [7]:
# Load model
model_name = "xlm-roberta-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)
teacher_model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

# Tokenize data
def tokenize_data(texts, tokenizer, max_length=128):
    encodings = tokenizer(texts, truncation=True, padding=True, max_length=max_length, return_tensors='pt')
    return encodings['input_ids'], encodings['attention_mask']

x_train_ids, x_train_mask = tokenize_data(x_train, tokenizer)
x_val_ids, x_val_mask = tokenize_data(x_val, tokenizer)

y_train_tensor = torch.tensor(y_train)
y_val_tensor = torch.tensor(y_val)

# Create dataloaders
train_dataset = TensorDataset(x_train_ids, x_train_mask, y_train_tensor)
train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)

val_dataset = TensorDataset(x_val_ids, x_val_mask, y_val_tensor)
val_loader = DataLoader(val_dataset, batch_size=8)

# Setup training
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
teacher_model.to(device)
optimizer = AdamW(teacher_model.parameters(), lr=1e-5)

# History to store metrics
history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}

# Training loop
for epoch in range(10):
    # Training phase
    teacher_model.train()
    train_loss = 0
    train_correct = 0
    train_total = 0
    
    for batch in train_loader:
        input_ids, attention_mask, labels = [x.to(device) for x in batch]
        
        outputs = teacher_model(input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss
        train_loss += loss.item()
        
        # Calculate accuracy
        logits = outputs.logits
        predictions = torch.argmax(logits, dim=1)
        train_correct += (predictions == labels).sum().item()
        train_total += labels.size(0)
        
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()
    
    # Validation phase
    teacher_model.eval()
    val_loss = 0
    val_correct = 0
    val_total = 0
    
    with torch.no_grad():
        for batch in val_loader:
            input_ids, attention_mask, labels = [x.to(device) for x in batch]
            
            outputs = teacher_model(input_ids, attention_mask=attention_mask, labels=labels)
            loss = outputs.loss
            val_loss += loss.item()
            
            logits = outputs.logits
            predictions = torch.argmax(logits, dim=1)
            val_correct += (predictions == labels).sum().item()
            val_total += labels.size(0)
    
    # Calculate averages
    avg_train_loss = train_loss / len(train_loader)
    avg_train_acc = train_correct / train_total
    avg_val_loss = val_loss / len(val_loader)
    avg_val_acc = val_correct / val_total
    
    # Store in history
    history['train_loss'].append(avg_train_loss)
    history['train_acc'].append(avg_train_acc)
    history['val_loss'].append(avg_val_loss)
    history['val_acc'].append(avg_val_acc)
    
    # Print progress
    print(f"Epoch {epoch+1}/10 | Train Loss: {avg_train_loss:.4f} | Train Acc: {avg_train_acc:.4f} | Val Loss: {avg_val_loss:.4f} | Val Acc: {avg_val_acc:.4f}")


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.10M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.12G [00:00<?, ?B/s]

Some weights of XLMRobertaForSequenceClassification were not initialized from the model checkpoint at xlm-roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch 1/10 | Train Loss: 0.6873 | Train Acc: 0.5481 | Val Loss: 0.6604 | Val Acc: 0.7323
Epoch 2/10 | Train Loss: 0.5343 | Train Acc: 0.7285 | Val Loss: 0.4375 | Val Acc: 0.7828
Epoch 3/10 | Train Loss: 0.5048 | Train Acc: 0.7926 | Val Loss: 0.4279 | Val Acc: 0.8232
Epoch 4/10 | Train Loss: 0.3748 | Train Acc: 0.8381 | Val Loss: 0.4152 | Val Acc: 0.8182
Epoch 5/10 | Train Loss: 0.3211 | Train Acc: 0.8786 | Val Loss: 0.4416 | Val Acc: 0.8081
Epoch 6/10 | Train Loss: 0.2644 | Train Acc: 0.9039 | Val Loss: 0.4117 | Val Acc: 0.8384
Epoch 7/10 | Train Loss: 0.2269 | Train Acc: 0.9089 | Val Loss: 0.4160 | Val Acc: 0.8131
Epoch 8/10 | Train Loss: 0.1864 | Train Acc: 0.9494 | Val Loss: 0.6237 | Val Acc: 0.8232
Epoch 9/10 | Train Loss: 0.1978 | Train Acc: 0.9545 | Val Loss: 0.6878 | Val Acc: 0.8081
Epoch 10/10 | Train Loss: 0.1556 | Train Acc: 0.9494 | Val Loss: 0.5393 | Val Acc: 0.8232


### Visualize Teacher Training History

Plot training and validation loss/accuracy curves using Plotly to assess convergence and detect overfitting. Visual inspection reveals whether the model is learning properly, overfitting (gap between train/val), underfitting (both high), or converging. This guides decisions about training duration and hyperparameters.

In [9]:
def plot_training_history(history, title="Training History"):
    """
    Plot training and validation loss/accuracy using Plotly
    
    Args:
        history: dict with keys 'train_loss', 'val_loss', 'train_acc', 'val_acc'
        title: title for the figure
    """
    
    # Create subplots
    fig = make_subplots(
        rows=1, cols=2,
        subplot_titles=("Loss", "Accuracy")
    )
    
    epochs = list(range(1, len(history['train_loss']) + 1))
    
    # Add loss traces
    fig.add_trace(
        go.Scatter(x=epochs, y=history['train_loss'], mode='lines+markers', 
                   name='Train Loss', line=dict(color='blue')),
        row=1, col=1
    )
    fig.add_trace(
        go.Scatter(x=epochs, y=history['val_loss'], mode='lines+markers', 
                   name='Val Loss', line=dict(color='red')),
        row=1, col=1
    )
    
    # Add accuracy traces
    fig.add_trace(
        go.Scatter(x=epochs, y=history['train_acc'], mode='lines+markers', 
                   name='Train Accuracy', line=dict(color='green')),
        row=1, col=2
    )
    fig.add_trace(
        go.Scatter(x=epochs, y=history['val_acc'], mode='lines+markers', 
                   name='Val Accuracy', line=dict(color='orange')),
        row=1, col=2
    )
    
    # Update layout
    fig.update_xaxes(title_text="Epoch", row=1, col=1)
    fig.update_xaxes(title_text="Epoch", row=1, col=2)
    fig.update_yaxes(title_text="Loss", row=1, col=1)
    fig.update_yaxes(title_text="Accuracy", row=1, col=2)
    
    fig.update_layout(height=400, width=1000, title_text=title, hovermode='x unified')
    
    fig.show()

# Use it
plot_training_history(history, title="Teacher Model Training History")

In [10]:
teacher_model

XLMRobertaForSequenceClassification(
  (roberta): XLMRobertaModel(
    (embeddings): XLMRobertaEmbeddings(
      (word_embeddings): Embedding(250002, 768, padding_idx=1)
      (position_embeddings): Embedding(514, 768, padding_idx=1)
      (token_type_embeddings): Embedding(1, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): XLMRobertaEncoder(
      (layer): ModuleList(
        (0-11): 12 x XLMRobertaLayer(
          (attention): XLMRobertaAttention(
            (self): XLMRobertaSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): XLMRobertaSelfOutput(
              (dense): Linear(in_features=768, out_features=

### Define Sentiment Prediction Function
This is a reusable function that accepts new Filipino reviews and returns sentiment predictions with confidence scores. This function encapsulates the inference pipeline, making it easy to test the model on new samples without repeating tokenization and model setup code. Used for both qualitative testing and batch inference.

In [11]:
def predict_sentiment(texts, model, tokenizer, device):
    model.eval()
    encodings = tokenizer(texts, truncation=True, padding=True, max_length=128, return_tensors='pt')
    input_ids = encodings['input_ids'].to(device)
    attention_mask = encodings['attention_mask'].to(device)
    
    with torch.no_grad():
        outputs = model(input_ids, attention_mask=attention_mask)
        logits = outputs.logits
        probabilities = torch.softmax(logits, dim=1)
        predictions = torch.argmax(logits, dim=1)
    
    sentiments = ['negative', 'positive']
    return [sentiments[p.item()] for p in predictions], probabilities.cpu().numpy()

In [12]:
# Test
sample_reviews = ["Magandang produkto!", "Napakasama ng quality", "ampangit", "basurang item"]
sentiments, scores = predict_sentiment(sample_reviews, teacher_model, tokenizer, device)

for review, sentiment, score in zip(sample_reviews, sentiments, scores):
    print(f"{review} -> {sentiment} (confidence: {score[1]:.4f})")

Magandang produkto! -> positive (confidence: 0.9763)
Napakasama ng quality -> negative (confidence: 0.1366)
ampangit -> negative (confidence: 0.0290)
basurang item -> negative (confidence: 0.0039)


### Generate Soft Targets from Teacher

Run the trained teacher model on all data splits (train/val/test) to extract probability distributions (soft targets) that encode the teacher's knowledge. Soft targets preserve the teacher's decision confidence and boundary information, which is more informative than hard labels for training the student. They enable knowledge distillation by transferring the teacher's learned patterns.

In [13]:
# Function to generate soft targets
def generate_soft_targets(model, data_loader, device):
    """
    Generate soft targets (probabilities) from teacher model
    
    Args:
        model: trained teacher model
        data_loader: DataLoader with input_ids, attention_mask
        device: cuda or cpu
    
    Returns:
        soft_targets: numpy array of shape (num_samples, num_classes)
    """
    model.eval()
    all_probabilities = []
    
    with torch.no_grad():
        for batch in data_loader:
            input_ids, attention_mask = batch[0].to(device), batch[1].to(device)
            
            outputs = model(input_ids, attention_mask=attention_mask)
            logits = outputs.logits
            
            # Convert logits to probabilities
            probabilities = torch.softmax(logits, dim=1)
            all_probabilities.append(probabilities.cpu().numpy())
    
    # Concatenate all batches
    soft_targets = np.concatenate(all_probabilities, axis=0)
    return soft_targets

# Generate soft targets for train, val, and test sets
print("Generating soft targets for training data...")
soft_targets_train = generate_soft_targets(teacher_model, train_loader, device)
print(f"Train soft targets shape: {soft_targets_train.shape}")
print(f"Sample soft target: {soft_targets_train[0]}")

print("\nGenerating soft targets for validation data...")
soft_targets_val = generate_soft_targets(teacher_model, val_loader, device)
print(f"Val soft targets shape: {soft_targets_val.shape}")

# Generate for test set (create test loader first)
print("\nTokenizing test data...")
x_test_ids, x_test_mask = tokenize_data(x_test, tokenizer)
y_test_tensor = torch.tensor(y_test)
test_dataset = TensorDataset(x_test_ids, x_test_mask, y_test_tensor)
test_loader = DataLoader(test_dataset, batch_size=8)

print("Generating soft targets for test data...")
soft_targets_test = generate_soft_targets(teacher_model, test_loader, device)
print(f"Test soft targets shape: {soft_targets_test.shape}")

# Save soft targets
print("\nSaving soft targets...")
np.save('soft_targets_train.npy', soft_targets_train)
np.save('soft_targets_val.npy', soft_targets_val)
np.save('soft_targets_test.npy', soft_targets_test)

Generating soft targets for training data...
Train soft targets shape: (593, 2)
Sample soft target: [0.00339477 0.9966053 ]

Generating soft targets for validation data...
Val soft targets shape: (198, 2)

Tokenizing test data...
Generating soft targets for test data...
Test soft targets shape: (198, 2)

Saving soft targets...


## Load Data and Train Student Model with Dual Loss

Load DistilBERT, resize vocabulary to match teacher, prepare data with soft targets + hard labels, and train for 15 epochs using combined distillation and supervised loss. Why DistilBERT. since it has 6 layers, 335M params and is 40% smaller than teacher (12 layers, 559M params). Dual loss balances learning from teacher knowledge (60% KL divergence) and ground truth (40% cross-entropy), preventing the student from perpetuating teacher errors while enabling knowledge transfer.

In [14]:
# Train student with BOTH soft targets AND hard labels
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from torch.utils.data import DataLoader, TensorDataset
import torch
from torch.optim import AdamW
import numpy as np

tokenizer = AutoTokenizer.from_pretrained("xlm-roberta-base")
student_model = AutoModelForSequenceClassification.from_pretrained("distilbert-base-multilingual-cased", num_labels=2)
student_model.resize_token_embeddings(len(tokenizer))

def tokenize_data(texts, tokenizer, max_length=128):
    encodings = tokenizer(texts, truncation=True, padding=True, max_length=max_length, return_tensors='pt')
    return encodings['input_ids'], encodings['attention_mask']

x_train_ids, x_train_mask = tokenize_data(x_train, tokenizer)
x_val_ids, x_val_mask = tokenize_data(x_val, tokenizer)

soft_targets_train = np.load('soft_targets_train.npy')
soft_targets_val = np.load('soft_targets_val.npy')
soft_targets_train = torch.tensor(soft_targets_train, dtype=torch.float32)
soft_targets_val = torch.tensor(soft_targets_val, dtype=torch.float32)

# Add hard labels
y_train_tensor = torch.tensor(y_train)
y_val_tensor = torch.tensor(y_val)

train_dataset = TensorDataset(x_train_ids, x_train_mask, soft_targets_train, y_train_tensor)
train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)

val_dataset = TensorDataset(x_val_ids, x_val_mask, soft_targets_val, y_val_tensor)
val_loader = DataLoader(val_dataset, batch_size=8)

device = torch.device('cuda')
student_model.to(device)
optimizer = AdamW(student_model.parameters(), lr=3e-5)

history = {'train_loss': [], 'val_loss': []}
temperature = 2.0
alpha = 0.6  # Weight for distillation loss vs supervised loss

# Training with both losses
for epoch in range(15):
    student_model.train()
    train_loss = 0
    
    for batch in train_loader:
        input_ids, attention_mask, soft_targets, hard_labels = [x.to(device) for x in batch]
        
        outputs = student_model(input_ids, attention_mask=attention_mask, labels=hard_labels)
        logits = outputs.logits
        supervised_loss = outputs.loss
        
        # Distillation loss
        student_probs = torch.softmax(logits / temperature, dim=1)
        teacher_probs = torch.softmax(soft_targets / temperature, dim=1)
        
        distill_loss = torch.nn.KLDivLoss(reduction='batchmean')(
            torch.log(student_probs),
            teacher_probs
        )
        
        # Combined loss
        loss = alpha * distill_loss + (1 - alpha) * supervised_loss
        train_loss += loss.item()
        
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()
    
    # Validation
    student_model.eval()
    val_loss = 0
    
    with torch.no_grad():
        for batch in val_loader:
            input_ids, attention_mask, soft_targets, hard_labels = [x.to(device) for x in batch]
            
            outputs = student_model(input_ids, attention_mask=attention_mask, labels=hard_labels)
            logits = outputs.logits
            supervised_loss = outputs.loss
            
            student_probs = torch.softmax(logits / temperature, dim=1)
            teacher_probs = torch.softmax(soft_targets / temperature, dim=1)
            
            distill_loss = torch.nn.KLDivLoss(reduction='batchmean')(
                torch.log(student_probs),
                teacher_probs
            )
            
            loss = alpha * distill_loss + (1 - alpha) * supervised_loss
            val_loss += loss.item()
    
    avg_train_loss = train_loss / len(train_loader)
    avg_val_loss = val_loss / len(val_loader)
    
    history['train_loss'].append(avg_train_loss)
    history['val_loss'].append(avg_val_loss)
    
    print(f"Epoch {epoch+1}/15 | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f}")

print("Student training with hard labels complete!")

config.json:   0%|          | 0.00/466 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/542M [00:00<?, ?B/s]

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-multilingual-cased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`


Epoch 1/15 | Train Loss: 0.2961 | Val Loss: 0.2922
Epoch 2/15 | Train Loss: 0.2893 | Val Loss: 0.2773
Epoch 3/15 | Train Loss: 0.2573 | Val Loss: 0.2755
Epoch 4/15 | Train Loss: 0.2046 | Val Loss: 0.2505
Epoch 5/15 | Train Loss: 0.1734 | Val Loss: 0.2966
Epoch 6/15 | Train Loss: 0.2020 | Val Loss: 0.2815
Epoch 7/15 | Train Loss: 0.1547 | Val Loss: 0.2683
Epoch 8/15 | Train Loss: 0.1517 | Val Loss: 0.3038
Epoch 9/15 | Train Loss: 0.1452 | Val Loss: 0.2927
Epoch 10/15 | Train Loss: 0.1453 | Val Loss: 0.2914
Epoch 11/15 | Train Loss: 0.1411 | Val Loss: 0.2532
Epoch 12/15 | Train Loss: 0.1422 | Val Loss: 0.2657
Epoch 13/15 | Train Loss: 0.1388 | Val Loss: 0.2676
Epoch 14/15 | Train Loss: 0.1373 | Val Loss: 0.2704
Epoch 15/15 | Train Loss: 0.1378 | Val Loss: 0.2820
Student training with hard labels complete!


### Evaluate and Compare Teacher vs Student

Test both teacher and student models on the held-out test set, compute multiple metrics (accuracy, precision, recall, F1), measure inference time, and compare performance. Comprehensive evaluation reveals the trade-off between model size/speed and accuracy. Inference time measurement demonstrates practical efficiency gains. Comparison metrics quantify distillation success and guide deployment decisions.

In [16]:


# Load test soft targets
soft_targets_test = np.load('/kaggle/working/soft_targets_test.npy')
soft_targets_test = torch.tensor(soft_targets_test, dtype=torch.float32)

# Create test loader
x_test_ids, x_test_mask = tokenize_data(x_test, tokenizer)
y_test_tensor = torch.tensor(y_test)

test_dataset = TensorDataset(x_test_ids, x_test_mask, y_test_tensor)
test_loader = DataLoader(test_dataset, batch_size=8)

# Move models to CPU
device = torch.device('cuda')
student_model.to(device)
teacher_model.to(device)

# Evaluation function with metrics
def evaluate_model_acc(model, data_loader, device):
    model.eval()
    all_predictions = []
    all_labels = []
    
    with torch.no_grad():
        for batch in data_loader:
            input_ids, attention_mask, labels = [x.to(device) for x in batch]
            
            outputs = model(input_ids, attention_mask=attention_mask)
            logits = outputs.logits
            predictions = torch.argmax(logits, dim=1)
            
            all_predictions.extend(predictions.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    
    accuracy = accuracy_score(all_labels, all_predictions)
    precision = precision_score(all_labels, all_predictions)
    recall = recall_score(all_labels, all_predictions)
    f1 = f1_score(all_labels, all_predictions)
    
    return {'accuracy': accuracy, 'precision': precision, 'recall': recall, 'f1': f1}

# Inference function for new reviews
def predict_sentiment(texts, model, tokenizer, device):
    model.eval()
    
    encodings = tokenizer(texts, truncation=True, padding=True, max_length=128, return_tensors='pt')
    input_ids = encodings['input_ids'].to(device)
    attention_mask = encodings['attention_mask'].to(device)
    
    with torch.no_grad():
        outputs = model(input_ids, attention_mask=attention_mask)
        logits = outputs.logits
        probabilities = torch.softmax(logits, dim=1)
        predictions = torch.argmax(logits, dim=1)
    
    sentiments = ['negative', 'positive']
    return [sentiments[p.item()] for p in predictions], probabilities.cpu().numpy()



### Model Evaluation Metrics: Definitions and Measurement

#### Accuracy

**What it does:** Measures the proportion of correct predictions (both positive and negative) out of all predictions made.

**How we measure it:** 
```
Accuracy = (True Positives + True Negatives) / Total Predictions
```

**Interpretation:** If the model correctly predicts 154 out of 198 test samples, accuracy is 77.78%. It answers: "How often is the model correct overall?"

**Limitation:** Does not distinguish between false positives and false negatives. A model predicting everything as positive could achieve high accuracy on imbalanced data.


#### Precision

**What it does:** Measures how many of the positive predictions are actually correct. It answers: "When the model says positive, how often is it right?"

**How we measure it:**
```
Precision = True Positives / (True Positives + False Positives)
```

**Interpretation:** If the model predicts 100 samples as positive and 83 are actually positive, precision is 83%. High precision means few false alarms.

**When to use:** Critical when false positives are costly (e.g., flagging non-spam as spam).


#### Recall

**What it does:** Measures how many actual positive cases the model identifies. It answers: "Of all the positive samples, how many did the model catch?"

**How we measure it:**
```
Recall = True Positives / (True Positives + False Negatives)
```

**Interpretation:** If there are 140 positive samples and the model identifies 101, recall is 72.14%. High recall means few missed cases.

**When to use:** Critical when false negatives are costly (e.g., missing disease diagnosis).


#### F1-Score

**What it does:** Combines precision and recall into a single metric. It is the harmonic mean of precision and recall.

**How we measure it:**
```
F1-Score = 2 × (Precision × Recall) / (Precision + Recall)
```

**Interpretation:** Ranges from 0 to 1, where 1 is perfect. Balances the trade-off between precision and recall. A model with high precision but low recall (or vice versa) will have lower F1-score.

**When to use:** Best for overall model performance when you care equally about both false positives and false negatives.


## Inference Time

**What it does:** Measures how long it takes the model to make predictions on a dataset. It is a measure of computational efficiency.

**How we measure it:**
```
Inference Time = Time to process entire test set / Number of batches
```

**Interpretation:** Teacher took 0.764 seconds to process 198 test samples. Student took 0.406 seconds. Lower is better.

**Why it matters:** Direct impact on deployment scalability. Faster models can serve more requests per unit time.


## Parameters

**What it does:** Counts the total number of learnable weights and biases in the model. Indicates model size and memory requirements.

**How we measure it:**
```
Parameters = Sum of all weight matrices + bias vectors
```

**Interpretation:** Teacher has 559 million parameters. Student has 335 million parameters (40% smaller). Fewer parameters mean less memory, storage, and faster loading.

**Why it matters:** Determines whether the model can run on mobile devices, edge devices, or resource-constrained servers.


## Knowledge Retention

**What it does:** Measures what percentage of the teacher's performance the student achieved after distillation.

**How we measure it:**
```
Knowledge Retention = (Student Accuracy / Teacher Accuracy) × 100
```

**Interpretation:** Student achieved 77.78% / 84.85% = 91.6% of teacher's accuracy. The student retained 91.6% of the teacher's knowledge despite 40% parameter reduction.

**Significance:** Above 90% retention is considered successful distillation. It quantifies the effectiveness of knowledge transfer.


## Speed Improvement

**What it does:** Measures how much faster the student model is compared to the teacher.

**How we measure it:**
```
Speed Improvement = Teacher Inference Time / Student Inference Time
```

**Interpretation:** 0.764s / 0.406s = 1.88x. The student processes predictions 1.88 times faster than the teacher.

**Practical meaning:** In production, the student can handle 1.88x more requests per second on the same hardware.

---

## Summary Table: What Each Metric Reveals

| Metric | Measures | Good When | Problem When |
|--------|----------|-----------|-------------|
| Accuracy | Overall correctness | High (>85%) | Ignores precision/recall imbalance |
| Precision | Quality of positive predictions | High (>85%) | Misses many positive cases (low recall) |
| Recall | Coverage of positive cases | High (>85%) | Makes many false positive errors |
| F1-Score | Balance of precision and recall | High (>0.85) | Either precision or recall is low |
| Inference Time | Speed of predictions | Low (<0.5s) | Model is slow for real-time use |
| Parameters | Model size and memory | Low (< Teacher) | Model too large to deploy |
| Knowledge Retention | Distillation effectiveness | High (>90%) | Student lost too much performance |
| Speed Improvement | Relative speedup | High (>1.5x) | Compression gained little speedup |

---

## How We Measured These in our Project

For the test set of 198 Filipino reviews:

1. Made predictions with both teacher and student models
2. Compared predictions against ground-truth labels
3. Calculated TP, TN, FP, FN for accuracy, precision, recall, F1
4. Timed inference using Python's time module
5. Counted model parameters using model.numel() or config files
6. Divided student accuracy by teacher accuracy for retention ratio
7. Divided teacher time by student time for speed improvement

All metrics were computed on the held-out test set to ensure unbiased evaluation.

In [23]:
import time
import numpy as np
import plotly.graph_objects as go
import plotly.subplots as sp
from plotly.subplots import make_subplots
from sklearn.metrics import (
    accuracy_score, 
    precision_score, 
    recall_score, 
    f1_score,
    confusion_matrix,
    classification_report
)

# Enhanced evaluation function with predictions tracking
def evaluate_model_acc(model, data_loader, device):
    model.eval()
    all_predictions = []
    all_labels = []
    
    with torch.no_grad():
        for batch in data_loader:
            input_ids, attention_mask, labels = [x.to(device) for x in batch]
            
            outputs = model(input_ids, attention_mask=attention_mask)
            logits = outputs.logits
            predictions = torch.argmax(logits, dim=1)
            
            all_predictions.extend(predictions.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    
    accuracy = accuracy_score(all_labels, all_predictions)
    precision = precision_score(all_labels, all_predictions)
    recall = recall_score(all_labels, all_predictions)
    f1 = f1_score(all_labels, all_predictions)
    
    return {
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'predictions': all_predictions,
        'labels': all_labels
    }

# Evaluate models
print("Evaluating student on test set...")
start_time = time.time()
student_metrics = evaluate_model_acc(student_model, test_loader, device)
student_time = time.time() - start_time

print(f"Student Accuracy: {student_metrics['accuracy']:.4f}")
print(f"Student Precision: {student_metrics['precision']:.4f}")
print(f"Student Recall: {student_metrics['recall']:.4f}")
print(f"Student F1: {student_metrics['f1']:.4f}")
print(f"Inference time: {student_time:.3f}s")

print("\n" + "="*50)
print("Evaluating teacher on test set...")
start_time = time.time()
teacher_metrics = evaluate_model_acc(teacher_model, test_loader, device)
teacher_time = time.time() - start_time

print(f"Teacher Accuracy: {teacher_metrics['accuracy']:.4f}")
print(f"Teacher Precision: {teacher_metrics['precision']:.4f}")
print(f"Teacher Recall: {teacher_metrics['recall']:.4f}")
print(f"Teacher F1: {teacher_metrics['f1']:.4f}")
print(f"Inference time: {teacher_time:.3f}s")

# Comparison
print("\n" + "="*50)
print("COMPARISON")
print("="*50)
print(f"Accuracy Gap: {abs(teacher_metrics['accuracy'] - student_metrics['accuracy']):.4f}")
print(f"F1 Gap: {abs(teacher_metrics['f1'] - student_metrics['f1']):.4f}")
print(f"Speed Improvement: {teacher_time / student_time:.2f}x faster" if student_time > 0 else "N/A")

# ============================================================
# PLOTLY VISUALIZATIONS
# ============================================================

# Create confusion matrices
student_cm = confusion_matrix(student_metrics['labels'], student_metrics['predictions'])
teacher_cm = confusion_matrix(teacher_metrics['labels'], teacher_metrics['predictions'])

# Convert to percentages
student_cm_pct = student_cm.astype('float') / student_cm.sum(axis=1)[:, np.newaxis] * 100
teacher_cm_pct = teacher_cm.astype('float') / teacher_cm.sum(axis=1)[:, np.newaxis] * 100

# Create text annotations with both count and percentage
student_cm_text = np.array([[f'{count}<br>({pct:.1f}%)' 
                            for count, pct in zip(row_count, row_pct)]
                           for row_count, row_pct in zip(student_cm, student_cm_pct)])

teacher_cm_text = np.array([[f'{count}<br>({pct:.1f}%)' 
                            for count, pct in zip(row_count, row_pct)]
                           for row_count, row_pct in zip(teacher_cm, teacher_cm_pct)])

# Create subplots
fig = make_subplots(
    rows=2, cols=3,
    subplot_titles=(
        'Student Model - Confusion Matrix',
        'Teacher Model - Confusion Matrix', 
        'Performance Metrics Comparison',
        'Inference Speed Comparison',
        'Performance Gap Analysis',
        'Summary Statistics'
    ),
    specs=[
        [{'type': 'heatmap'}, {'type': 'heatmap'}, {'type': 'bar'}],
        [{'type': 'bar'}, {'type': 'bar'}, {'type': 'table'}]
    ],
    vertical_spacing=0.12,
    horizontal_spacing=0.10
)

# 1. Student Confusion Matrix (with percentages)
fig.add_trace(
    go.Heatmap(
        z=student_cm_pct,
        x=['Negative', 'Positive'],
        y=['Negative', 'Positive'],
        text=student_cm_text,
        texttemplate='%{text}',
        textfont={"size": 14},
        colorscale='Blues',
        showscale=True,
        colorbar=dict(title="Percentage", x=0.31, len=0.38, y=0.77),
        hovertemplate='True: %{y}<br>Predicted: %{x}<br>Percentage: %{z:.1f}%<extra></extra>',
        zmin=0,
        zmax=100
    ),
    row=1, col=1
)

# 2. Teacher Confusion Matrix (with percentages)
fig.add_trace(
    go.Heatmap(
        z=teacher_cm_pct,
        x=['Negative', 'Positive'],
        y=['Negative', 'Positive'],
        text=teacher_cm_text,
        texttemplate='%{text}',
        textfont={"size": 14},
        colorscale='Greens',
        showscale=True,
        colorbar=dict(title="Percentage", x=0.64, len=0.38, y=0.77),
        hovertemplate='True: %{y}<br>Predicted: %{x}<br>Percentage: %{z:.1f}%<extra></extra>',
        zmin=0,
        zmax=100
    ),
    row=1, col=2
)

# 3. Metrics Comparison
metrics_names = ['Accuracy', 'Precision', 'Recall', 'F1']
student_scores = [
    student_metrics['accuracy'],
    student_metrics['precision'],
    student_metrics['recall'],
    student_metrics['f1']
]
teacher_scores = [
    teacher_metrics['accuracy'],
    teacher_metrics['precision'],
    teacher_metrics['recall'],
    teacher_metrics['f1']
]

fig.add_trace(
    go.Bar(
        name='Student',
        x=metrics_names,
        y=student_scores,
        marker_color='#3498db',
        text=[f'{s:.3f}' for s in student_scores],
        textposition='outside',
        hovertemplate='%{x}<br>Score: %{y:.4f}<extra></extra>'
    ),
    row=1, col=3
)

fig.add_trace(
    go.Bar(
        name='Teacher',
        x=metrics_names,
        y=teacher_scores,
        marker_color='#2ecc71',
        text=[f'{s:.3f}' for s in teacher_scores],
        textposition='outside',
        hovertemplate='%{x}<br>Score: %{y:.4f}<extra></extra>'
    ),
    row=1, col=3
)

# 4. Inference Time Comparison
fig.add_trace(
    go.Bar(
        x=[teacher_time, student_time],
        y=['Teacher', 'Student'],
        orientation='h',
        marker_color=['#2ecc71', '#3498db'],
        text=[f'{teacher_time:.2f}s', f'{student_time:.2f}s'],
        textposition='outside',
        showlegend=False,
        hovertemplate='%{y}<br>Time: %{x:.3f}s<extra></extra>'
    ),
    row=2, col=1
)

# 5. Performance Gap Analysis
gaps = [
    abs(teacher_metrics['accuracy'] - student_metrics['accuracy']),
    abs(teacher_metrics['precision'] - student_metrics['precision']),
    abs(teacher_metrics['recall'] - student_metrics['recall']),
    abs(teacher_metrics['f1'] - student_metrics['f1'])
]

gap_colors = ['#e74c3c' if gap > 0.02 else '#f39c12' if gap > 0.01 else '#27ae60' for gap in gaps]

fig.add_trace(
    go.Bar(
        x=metrics_names,
        y=gaps,
        marker_color=gap_colors,
        text=[f'{g:.4f}' for g in gaps],
        textposition='outside',
        showlegend=False,
        hovertemplate='%{x}<br>Gap: %{y:.4f}<extra></extra>'
    ),
    row=2, col=2
)

# 6. Summary Table
summary_data = [
    ['<b>Metric</b>', '<b>Student</b>', '<b>Teacher</b>', '<b>Gap</b>'],
    ['Accuracy', f"{student_metrics['accuracy']:.4f}", f"{teacher_metrics['accuracy']:.4f}", 
     f"{abs(teacher_metrics['accuracy'] - student_metrics['accuracy']):.4f}"],
    ['Precision', f"{student_metrics['precision']:.4f}", f"{teacher_metrics['precision']:.4f}",
     f"{abs(teacher_metrics['precision'] - student_metrics['precision']):.4f}"],
    ['Recall', f"{student_metrics['recall']:.4f}", f"{teacher_metrics['recall']:.4f}",
     f"{abs(teacher_metrics['recall'] - student_metrics['recall']):.4f}"],
    ['F1 Score', f"{student_metrics['f1']:.4f}", f"{teacher_metrics['f1']:.4f}",
     f"{abs(teacher_metrics['f1'] - student_metrics['f1']):.4f}"],
    ['Time (s)', f"{student_time:.2f}", f"{teacher_time:.2f}",
     f"{teacher_time/student_time:.2f}x faster"]
]

fig.add_trace(
    go.Table(
        header=dict(
            values=summary_data[0],
            fill_color='#34495e',
            align='center',
            font=dict(color='white', size=12, family='Arial Black')
        ),
        cells=dict(
            values=list(zip(*summary_data[1:])),
            fill_color=[['#ecf0f1', 'white'] * 3],
            align='center',
            font=dict(size=11)
        )
    ),
    row=2, col=3
)

# Update layout
fig.update_xaxes(title_text="Predicted Label", row=1, col=1)
fig.update_yaxes(title_text="True Label", row=1, col=1)
fig.update_xaxes(title_text="Predicted Label", row=1, col=2)
fig.update_yaxes(title_text="True Label", row=1, col=2)
fig.update_xaxes(title_text="Metric", row=1, col=3)
fig.update_yaxes(title_text="Score", range=[0.7, 0.95], row=1, col=3)
fig.update_xaxes(title_text="Inference Time (seconds)", row=2, col=1)
fig.update_yaxes(title_text="Model", row=2, col=1)
fig.update_xaxes(title_text="Metric", row=2, col=2)
fig.update_yaxes(title_text="Gap (Teacher - Student)", row=2, col=2)

fig.update_layout(
    title_text='<b>Knowledge Distillation: Student vs Teacher Model Evaluation</b>',
    title_x=0.5,
    title_font_size=20,
    showlegend=True,
    height=900,
    width=1600,
    hovermode='closest',
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=1.02,
        xanchor="right",
        x=0.65
    )
)

# Show the figure
fig.show()

# Save as HTML (interactive)
fig.write_html('model_evaluation_interactive.html')

# Save as static image (requires kaleido)
try:
    fig.write_image('model_evaluation_comprehensive.png', width=1600, height=900)
    print("\nStatic image saved as 'model_evaluation_comprehensive.png'")
except:
    print("\nNote: Install kaleido to save static images: pip install kaleido")

print("Interactive HTML saved as 'model_evaluation_interactive.html'")

# Print Classification Reports
print("\n" + "="*50)
print("STUDENT MODEL - CLASSIFICATION REPORT")
print("="*50)
print(classification_report(student_metrics['labels'], student_metrics['predictions'],
                          target_names=['Negative', 'Positive']))

print("\n" + "="*50)
print("TEACHER MODEL - CLASSIFICATION REPORT")
print("="*50)
print(classification_report(teacher_metrics['labels'], teacher_metrics['predictions'],
                          target_names=['Negative', 'Positive']))

Evaluating student on test set...
Student Accuracy: 0.7778
Student Precision: 0.8152
Student Recall: 0.7353
Student F1: 0.7732
Inference time: 0.821s

Evaluating teacher on test set...
Teacher Accuracy: 0.8384
Teacher Precision: 0.8889
Teacher Recall: 0.7843
Teacher F1: 0.8333
Inference time: 1.366s

COMPARISON
Accuracy Gap: 0.0606
F1 Gap: 0.0601
Speed Improvement: 1.66x faster



Note: Install kaleido to save static images: pip install kaleido
Interactive HTML saved as 'model_evaluation_interactive.html'

STUDENT MODEL - CLASSIFICATION REPORT
              precision    recall  f1-score   support

    Negative       0.75      0.82      0.78        96
    Positive       0.82      0.74      0.77       102

    accuracy                           0.78       198
   macro avg       0.78      0.78      0.78       198
weighted avg       0.78      0.78      0.78       198


TEACHER MODEL - CLASSIFICATION REPORT
              precision    recall  f1-score   support

    Negative       0.80      0.90      0.84        96
    Positive       0.89      0.78      0.83       102

    accuracy                           0.84       198
   macro avg       0.84      0.84      0.84       198
weighted avg       0.84      0.84      0.84       198



In [25]:
# new reviews
sample_reviews = [
    "Magandang produkto, napakabilis ng delivery!",
    "Napakasama ng quality, disappointed ako",
    "Sulit na sulit, highly recommended!"
]

print("\nStudent Predictions:")
sentiments, scores = predict_sentiment(sample_reviews, student_model, tokenizer, device)
for review, sentiment, score in zip(sample_reviews, sentiments, scores):
    print(f"Review: {review}")
    print(f"Prediction: {sentiment} (confidence: {max(score):.4f})\n")

print("Teacher Predictions:")
sentiments, scores = predict_sentiment(sample_reviews, teacher_model, tokenizer, device)
for review, sentiment, score in zip(sample_reviews, sentiments, scores):
    print(f"Review: {review}")
    print(f"Prediction: {sentiment} (confidence: {max(score):.4f})\n")


Student Predictions:
Review: Magandang produkto, napakabilis ng delivery!
Prediction: positive (confidence: 0.8341)

Review: Napakasama ng quality, disappointed ako
Prediction: negative (confidence: 0.8366)

Review: Sulit na sulit, highly recommended!
Prediction: negative (confidence: 0.7761)

Teacher Predictions:
Review: Magandang produkto, napakabilis ng delivery!
Prediction: positive (confidence: 0.9982)

Review: Napakasama ng quality, disappointed ako
Prediction: negative (confidence: 0.9372)

Review: Sulit na sulit, highly recommended!
Prediction: positive (confidence: 0.9706)



## Define Model Save Function
The following is a reusable function that exports both model weights and tokenizer to disk in standard HuggingFace format. Standardized export format enables easy loading, sharing, and deployment of models without re-training. Includes both weights and tokenizer to ensure reproducibility.

In [26]:

def save_model(model, tokenizer, save_path, model_name):
    """
    Save model and tokenizer to disk
    
    Args:
        model: Trained transformer model
        tokenizer: Tokenizer used for preprocessing
        save_path: Directory path to save model (e.g., '/kaggle/working/')
        model_name: Name for the model (e.g., 'teacher_model' or 'student_model')
    """
    # Create model directory if it doesn't exist
    model_dir = os.path.join(save_path, model_name)
    os.makedirs(model_dir, exist_ok=True)
    
    # Save model weights and config
    model.save_pretrained(model_dir)
    print(f"✓ Model saved to {model_dir}")
    
    # Save tokenizer
    tokenizer.save_pretrained(model_dir)
    print(f"✓ Tokenizer saved to {model_dir}")

In [27]:
import os 

print("Saving teacher model...")
save_model(teacher_model, tokenizer, '/kaggle/working/', 'teacher_model')

print("\nSaving student model...")
save_model(student_model, tokenizer, '/kaggle/working/', 'student_model')

Saving teacher model...
✓ Model saved to /kaggle/working/teacher_model
✓ Tokenizer saved to /kaggle/working/teacher_model

Saving student model...
✓ Model saved to /kaggle/working/student_model
✓ Tokenizer saved to /kaggle/working/student_model


In [28]:
teacher_model

XLMRobertaForSequenceClassification(
  (roberta): XLMRobertaModel(
    (embeddings): XLMRobertaEmbeddings(
      (word_embeddings): Embedding(250002, 768, padding_idx=1)
      (position_embeddings): Embedding(514, 768, padding_idx=1)
      (token_type_embeddings): Embedding(1, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): XLMRobertaEncoder(
      (layer): ModuleList(
        (0-11): 12 x XLMRobertaLayer(
          (attention): XLMRobertaAttention(
            (self): XLMRobertaSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): XLMRobertaSelfOutput(
              (dense): Linear(in_features=768, out_features=

In [29]:
student_model


DistilBertForSequenceClassification(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(250002, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSdpaAttention(
            (dropout): Dropout(p=0.1, inplace=False)
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0.1, inplace=False)